# Notebook 3: Đánh giá phát hiện bất thường

Sử dụng trajectory đã có và tính precision/recall/f1.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from pathlib import Path
import json
import pandas as pd
import torch

from smarthome.config import load_config
from smarthome.models import GRUAutoencoder
from smarthome.dataset import AnomalyFeatureBuilder, SequenceDataset
from smarthome.anomaly import OnlineAnomalyDetector, fuse_alert
from smarthome.metrics import binary_anomaly_metrics

In [ ]:
ROOT = Path(__file__).resolve().parents[1]
cfg = load_config(ROOT / "config.yaml")
events = pd.read_csv(ROOT / "data" / "anomaly" / "location_events.csv")

In [ ]:
threshold_path = ROOT / "models" / "anomaly_threshold.json"
with threshold_path.open("r", encoding="utf-8") as f:
    threshold = float(json.load(f)["threshold"])

builder = AnomalyFeatureBuilder(cfg["rooms"], float(cfg["data"].get("anomaly_feature_max_dwell_minutes", 180)))
anomaly_cfg = cfg["anomaly_model"]
model = GRUAutoencoder(
    feature_dim=builder.feature_dim,
    hidden_size=int(anomaly_cfg["hidden_size"]),
    latent_size=int(anomaly_cfg["latent_size"]),
    num_layers=int(anomaly_cfg["num_layers"]),
)
model.load_state_dict(torch.load(ROOT / "models" / "gru_autoencoder.pt", map_location="cpu"))
model.eval()

In [ ]:
seq_len = int(cfg["data"]["anomaly_sequence_length"])
detector = OnlineAnomalyDetector(
    model=model,
    feature_builder=builder,
    sequence_length=seq_len,
    threshold=threshold,
    room_loss_weight=float(anomaly_cfg["room_loss_weight"]),
    continuous_loss_weight=float(anomaly_cfg["continuous_loss_weight"]),
    device="cpu",
)

In [ ]:
scores, labels = [], []
for _, row in events.iterrows():
    score = detector.update(
        room=row["room"],
        dwell_minutes=float(row["dwell_minutes"]),
        hour=float(row["hour"]),
        confidence=float(row.get("confidence", 1.0)),
    )
    if score is not None:
        scores.append(score)
        labels.append(int(row.get("is_anomaly", 0)))

scores = pd.Series(scores)
labels = pd.Series(labels)
preds = (scores > threshold).astype(int)
print(binary_anomaly_metrics(labels, preds))
print("Threshold:", threshold)